1. Simple Python wrapper

In [ ]:
import subprocess

cmd = [
    "pdal", "translate",
    "/data/beaufront/cloud1534c140323c0e17.las",
    "/data/beaufront/cloud1534c140323c0e17.copc.laz",
    "writers.copc"
]

subprocess.run(cmd, check=True)

2. Cleaner reusable function

In [ ]:
import subprocess
from pathlib import Path

def las_to_copc(input_las):
    input_las = Path(input_las)
    output_copc = input_las.with_suffix(".copc.laz")

    cmd = [
        "pdal", "translate",
        str(input_las),
        str(output_copc),
        "writers.copc"
    ]

    subprocess.run(cmd, check=True)

    return output_copc


las_to_copc("/data/beaufront/cloud1534c140323c0e17.las")

3. Batch convert a directory (useful for LiDAR surveys)

In [ ]:
import subprocess
from pathlib import Path

folder = Path("/data/beaufront")

for las in folder.glob("*.las"):
    out = las.with_suffix(".copc.laz")

    subprocess.run([
        "pdal", "translate",
        str(las),
        str(out),
        "writers.copc"
    ], check=True)

    print(f"Converted: {las.name}")

1️⃣ Read LAS metadata in Python

In [1]:
import pdal
import json

pipeline = {
    "pipeline": [
        "/data/beaufront/cloud1534c140323c0e17.las"
    ]
}

p = pdal.Pipeline(json.dumps(pipeline))
p.execute()

meta = p.metadata   # already a dict

print(json.dumps(meta, indent=2))

{
  "metadata": {
    "filters.merge": {},
    "readers.las": {
      "comp_spatialreference": "PROJCS[\"GDA2020 / MGA zone 55\",GEOGCS[\"GDA2020\",DATUM[\"Geocentric_Datum_of_Australia_2020\",SPHEROID[\"GRS 1980\",6378137,298.257222101,AUTHORITY[\"EPSG\",\"7019\"]],AUTHORITY[\"EPSG\",\"1168\"]],PRIMEM[\"Greenwich\",0,AUTHORITY[\"EPSG\",\"8901\"]],UNIT[\"degree\",0.0174532925199433,AUTHORITY[\"EPSG\",\"9122\"]],AUTHORITY[\"EPSG\",\"7844\"]],PROJECTION[\"Transverse_Mercator\"],PARAMETER[\"latitude_of_origin\",0],PARAMETER[\"central_meridian\",147],PARAMETER[\"scale_factor\",0.9996],PARAMETER[\"false_easting\",500000],PARAMETER[\"false_northing\",10000000],UNIT[\"metre\",1,AUTHORITY[\"EPSG\",\"9001\"]],AXIS[\"Easting\",EAST],AXIS[\"Northing\",NORTH],AUTHORITY[\"EPSG\",\"7855\"]]",
      "compressed": false,
      "copc": false,
      "count": 78833222,
      "creation_doy": 69,
      "creation_year": 2023,
      "dataformat_id": 3,
      "dataoffset": 1118,
      "filesource_id": 0,
    

In [2]:
import pdal
import json

pipeline = {
    "pipeline": [
        "/data/beaufront/20240312_SASMDD0001_2.copc.laz"
    ]
}

p = pdal.Pipeline(json.dumps(pipeline))
p.execute()

meta = p.metadata   # already a dict

print(json.dumps(meta, indent=2))

{
  "metadata": {
    "filters.merge": {},
    "readers.copc": {
      "comp_spatialreference": "COMPD_CS[\"GDA2020 / MGA zone 54 + AHD height\",PROJCS[\"GDA2020 / MGA zone 54\",GEOGCS[\"GDA2020\",DATUM[\"Geocentric_Datum_of_Australia_2020\",SPHEROID[\"GRS 1980\",6378137,298.257222101,AUTHORITY[\"EPSG\",\"7019\"]],AUTHORITY[\"EPSG\",\"1168\"]],PRIMEM[\"Greenwich\",0,AUTHORITY[\"EPSG\",\"8901\"]],UNIT[\"degree\",0.0174532925199433,AUTHORITY[\"EPSG\",\"9122\"]],AUTHORITY[\"EPSG\",\"7844\"]],PROJECTION[\"Transverse_Mercator\"],PARAMETER[\"latitude_of_origin\",0],PARAMETER[\"central_meridian\",141],PARAMETER[\"scale_factor\",0.9996],PARAMETER[\"false_easting\",500000],PARAMETER[\"false_northing\",10000000],UNIT[\"metre\",1,AUTHORITY[\"EPSG\",\"9001\"]],AXIS[\"Easting\",EAST],AXIS[\"Northing\",NORTH],AUTHORITY[\"EPSG\",\"7854\"]],VERT_CS[\"AHD height\",VERT_DATUM[\"Australian Height Datum\",2005,AUTHORITY[\"EPSG\",\"5111\"]],UNIT[\"metre\",1,AUTHORITY[\"EPSG\",\"9001\"]],AXIS[\"Gravity-rela

In [ ]:
import pdal
import json

pipeline = {
    "pipeline": [
        {
            "type": "readers.las",
            "filename": "input.las"
        },
        {
            "type": "filters.reprojection",
            "in_srs": "EPSG:4326+5711",
            "out_srs": "EPSG:7855+5711"
        },
        {
            "type": "writers.las",
            "filename": "output_gda2020.las"
        }
    ]
}

pipeline_json = json.dumps(pipeline)

pipe = pdal.Pipeline(pipeline_json)
pipe.execute()

print("Reprojection finished")

In [8]:
reader_meta = meta["metadata"]["readers.las"]

print(reader_meta.keys())

dict_keys(['comp_spatialreference', 'compressed', 'copc', 'count', 'creation_doy', 'creation_year', 'dataformat_id', 'dataoffset', 'filesource_id', 'global_encoding', 'global_encoding_base64', 'gtiff', 'header_size', 'major_version', 'maxx', 'maxy', 'maxz', 'minor_version', 'minx', 'miny', 'minz', 'offset_x', 'offset_y', 'offset_z', 'point_length', 'project_id', 'scale_x', 'scale_y', 'scale_z', 'software_id', 'spatialreference', 'srs', 'system_id', 'vlr_0', 'vlr_1', 'vlr_2'])


In [9]:
reader_meta = meta["metadata"]["readers.las"]

print("Point count:", reader_meta["count"])

bounds = {
    "minx": reader_meta["minx"],
    "miny": reader_meta["miny"],
    "minz": reader_meta["minz"],
    "maxx": reader_meta["maxx"],
    "maxy": reader_meta["maxy"],
    "maxz": reader_meta["maxz"],
}

print("Bounds:", bounds)
print("CRS:", reader_meta["srs"])

Point count: 78833222
Bounds: {'minx': 545386.7055, 'miny': 5344151.684, 'minz': 242.0586124, 'maxx': 545626.772, 'maxy': 5344481.973, 'maxz': 297.7471732}
CRS: {'compoundwkt': 'PROJCS["GDA2020 / MGA zone 55",GEOGCS["GDA2020",DATUM["Geocentric_Datum_of_Australia_2020",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1168"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","7844"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",147],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",10000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","7855"]]', 'horizontal': 'PROJCS["GDA2020 / MGA zone 55",GEOGCS["GDA2020",DATUM["Geocentric_Datum_of_Australia_2020",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["E

In [10]:
reader_meta = meta["metadata"]["readers.las"]

print(f"Point count: {reader_meta['count']}")
print(
    f"Bounds: "
    f"X[{reader_meta['minx']}, {reader_meta['maxx']}], "
    f"Y[{reader_meta['miny']}, {reader_meta['maxy']}], "
    f"Z[{reader_meta['minz']}, {reader_meta['maxz']}]"
)
print("CRS:", reader_meta["srs"])

Point count: 78833222
Bounds: X[545386.7055, 545626.772], Y[5344151.684, 5344481.973], Z[242.0586124, 297.7471732]
CRS: {'compoundwkt': 'PROJCS["GDA2020 / MGA zone 55",GEOGCS["GDA2020",DATUM["Geocentric_Datum_of_Australia_2020",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1168"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","7844"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",147],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",10000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","7855"]]', 'horizontal': 'PROJCS["GDA2020 / MGA zone 55",GEOGCS["GDA2020",DATUM["Geocentric_Datum_of_Australia_2020",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1168"]],PRIMEM["Greenwich",0,AUTHOR

In [ ]:
import subprocess
import json
from pathlib import Path
from datetime import datetime, timedelta

def las_to_copc_reproject(input_las, site_id="TCATNM0006"):
    input_las = Path(input_las)

    # Extract metadata
    result = subprocess.run(
        ["pdal", "info", str(input_las), "--metadata"],
        capture_output=True,
        text=True,
        check=True
    )

    meta = json.loads(result.stdout)
    md = meta["metadata"]

    year = md["creation_year"]
    doy = md["creation_doy"]

    date_obj = datetime(year, 1, 1) + timedelta(days=doy - 1)
    date_str = date_obj.strftime("%Y%m%d")

    output_copc = input_las.parent / f"{date_str}_{site_id}.copc.laz"

    cmd = [
        "pdal", "translate",
        "--writer", "writers.copc",
        str(input_las),
        str(output_copc),
        "reprojection",
        '--filters.reprojection.in_srs=EPSG:4326+5711',
        '--filters.reprojection.out_srs=EPSG:7855+5711'
    ]

    subprocess.run(cmd, check=True)

    return output_copc


las_to_copc_reproject("/data/beaufront/cloud1534c140323c0e17.las")

Return code: 1

STDOUT:
 

STDERR:
 PDAL: Couldn't create filter stage of type 'filters.writers.copc'.
You probably have a version of PDAL that didn't come with a plugin
you're trying to load.  Please see the FAQ at https://pdal.io/faq.html




RuntimeError: PDAL translate failed

In [ ]:
# (fvc_structure) nvidia@brev-rvbt83sj0:~/projects/fvc_structure/ForestFormer3D$ pdal translate \
#   /data/beaufront/cloud1534c140323c0e17.las \
#   /data/beaufront/20230310_TCATNM0006.copc.laz


In [ ]:
/data/beaufront/AHD/beaufront_metadata.json

In [ ]:
# 1) Rebuild COPC in AHD path
# pdal translate \
#   --writer writers.copc \
#   /data/beaufront/WGS84/cloud1534c140323c0e17.las \
#   /data/beaufront/AHD/20230310_TCATNM0006.copc.laz \
#   --writers.copc.a_srs="EPSG:7855+5711"

# # 2) Check metadata
# pdal info /data/beaufront/AHD/20230310_TCATNM0006.copc.laz --metadata

In [1]:
import json
import base64
import os
from pathlib import Path
import pdal

src = Path("/data/beaufront/AHD/20230310_TCATNM0006.copc.laz")
tmp = Path("/data/beaufront/AHD/20230310_TCATNM0006.tmp.copc.laz")
meta_json = Path("/home/nvidia/projects/fvc_structure/phase_2_lidar_coregistration/step_0_checks/beaufront_metadata.json")

payload_b64 = base64.b64encode(meta_json.read_bytes()).decode("ascii")

pipeline = {
    "pipeline": [
        {
            "type": "readers.copc",
            "filename": str(src)
        },
        {
            "type": "writers.copc",
            "filename": str(tmp),
            "forward": "all",
            "a_srs": "EPSG:7855+5711",
            "system_id": "UTAS TerraLuma - DJI Zenmuse L1",
            "software_id": "DJI Terra 3.5.0 + PDAL 2.8.4",
            "vlrs": [
                {
                    "user_id": "UTAS_TERRALUMA",
                    "record_id": 1,
                    "description": "Investigator: Laura N. Sotomayor",
                    "evlr": True,
                    "data": payload_b64
                }
            ]
        }
    ]
}

p = pdal.Pipeline(json.dumps(pipeline))
p.execute()
os.replace(tmp, src)
print(f"Updated: {src}")

Updated: /data/beaufront/AHD/20230310_TCATNM0006.copc.laz


In [ ]:
#  pdal info /data/beaufront/AHD/20230310_TCATNM0006.copc.laz --metadata

In [2]:
import subprocess
import json

result = subprocess.run(
    ["pdal", "info", "/data/beaufront/AHD/20230310_TCATNM0006.copc.laz", "--metadata"],
    capture_output=True,
    text=True,
    check=True
)

meta = json.loads(result.stdout)
print(json.dumps(meta, indent=2))

{
  "file_size": 566076574,
  "filename": "/data/beaufront/AHD/20230310_TCATNM0006.copc.laz",
  "metadata": {
    "comp_spatialreference": "COMPD_CS[\"GDA2020 / MGA zone 55 + AHD height\",PROJCS[\"GDA2020 / MGA zone 55\",GEOGCS[\"GDA2020\",DATUM[\"Geocentric_Datum_of_Australia_2020\",SPHEROID[\"GRS 1980\",6378137,298.257222101,AUTHORITY[\"EPSG\",\"7019\"]],AUTHORITY[\"EPSG\",\"1168\"]],PRIMEM[\"Greenwich\",0,AUTHORITY[\"EPSG\",\"8901\"]],UNIT[\"degree\",0.0174532925199433,AUTHORITY[\"EPSG\",\"9122\"]],AUTHORITY[\"EPSG\",\"7844\"]],PROJECTION[\"Transverse_Mercator\"],PARAMETER[\"latitude_of_origin\",0],PARAMETER[\"central_meridian\",147],PARAMETER[\"scale_factor\",0.9996],PARAMETER[\"false_easting\",500000],PARAMETER[\"false_northing\",10000000],UNIT[\"metre\",1,AUTHORITY[\"EPSG\",\"9001\"]],AXIS[\"Easting\",EAST],AXIS[\"Northing\",NORTH],AUTHORITY[\"EPSG\",\"7855\"]],VERT_CS[\"AHD height\",VERT_DATUM[\"Australian Height Datum\",2005,AUTHORITY[\"EPSG\",\"5111\"]],UNIT[\"metre\",1,AUTHOR

In [3]:
import base64
import json

# Extract and decode the VLR data
vlr_data_b64 = meta["metadata"]["vlr_2"]["data"]
metadata_json = base64.b64decode(vlr_data_b64).decode("utf-8")
metadata_dict = json.loads(metadata_json)

print(json.dumps(metadata_dict, indent=2))

{
  "ProjectName": "TerraLuma Research Lab",
  "SiteLocation": "TCATNM0006 (Beaufront)",
  "AcquisitionDate": "2023-03-09",
  "AcquisitionTime": "09:43:43",
  "Investigator": "Laura Natali Sotomayor",
  "Organisation": "University of Tasmania",
  "ContactEmail": "laura.sotomayor@utas.edu.au",
  "Platform": "DJI Matrice 300 (M300) RTK UAS",
  "SensorName": "DJI Zenmuse L1",
  "SensorType": "LiDAR",
  "FlightAltitude": "50 m above ground level",
  "FlightSpeed": "5 m/s",
  "FlightID": "TCATNM0006",
  "Description": "UAS LiDAR point cloud acquired at the Beaufront site using a DJI Zenmuse L1 sensor mounted on a DJI Matrice 300 RTK platform.",
  "ProcessingLevel": "Level 1",
  "ProcessingDate": "2023-03-22",
  "SoftwareUsed": "DJI Terra 3.5.0 and PDAL 2.8.4",
  "CRS": "EPSG:7855 (GDA2020 / MGA Zone 55) + EPSG:5711 (AHD height)",
  "FileFormat": "COPC LAZ"
}


In [4]:
import subprocess
import json
import base64

# Read metadata from the COPC file
result = subprocess.run(
    ["pdal", "info", "/data/beaufront/AHD/20230310_TCATNM0006.copc.laz", "--metadata"],
    capture_output=True,
    text=True,
    check=True
)

meta = json.loads(result.stdout)

# Extract and decode VLR #2 (your custom metadata)
vlr_data_b64 = meta["metadata"]["vlr_2"]["data"]
metadata_json = base64.b64decode(vlr_data_b64).decode("utf-8")
metadata_dict = json.loads(metadata_json)

print("🔍 Extracted Metadata from COPC file:")
print(json.dumps(metadata_dict, indent=2))

# Verify accuracy fields exist
if "CoordinateAccuracy" in metadata_dict:
    print(f"\n✅ CoordinateAccuracy: {metadata_dict['CoordinateAccuracy']}")
    print(f"✅ HorizontalAccuracy: {metadata_dict['HorizontalAccuracy']}")
    print(f"✅ VerticalAccuracy: {metadata_dict['VerticalAccuracy']}")
else:
    print("\n❌ Accuracy fields NOT found - need to update metadata")

🔍 Extracted Metadata from COPC file:
{
  "ProjectName": "TerraLuma Research Lab",
  "SiteLocation": "TCATNM0006 (Beaufront)",
  "AcquisitionDate": "2023-03-09",
  "AcquisitionTime": "09:43:43",
  "Investigator": "Laura Natali Sotomayor",
  "Organisation": "University of Tasmania",
  "ContactEmail": "laura.sotomayor@utas.edu.au",
  "Platform": "DJI Matrice 300 (M300) RTK UAS",
  "SensorName": "DJI Zenmuse L1",
  "SensorType": "LiDAR",
  "FlightAltitude": "50 m above ground level",
  "FlightSpeed": "5 m/s",
  "FlightID": "TCATNM0006",
  "Description": "UAS LiDAR point cloud acquired at the Beaufront site using a DJI Zenmuse L1 sensor mounted on a DJI Matrice 300 RTK platform.",
  "ProcessingLevel": "Level 1",
  "ProcessingDate": "2023-03-22",
  "SoftwareUsed": "DJI Terra 3.5.0 and PDAL 2.8.4",
  "CRS": "EPSG:7855 (GDA2020 / MGA Zone 55) + EPSG:5711 (AHD height)",
  "FileFormat": "COPC LAZ"
}

❌ Accuracy fields NOT found - need to update metadata


In [11]:
meta = {
    "Dataset": {
        "SiteLocation": "TCATNM0006 (Beaufront)",
        "ProjectName": "TerraLuma Research Lab",
        "DataProvider": "University of Tasmania"
    },

    "Acquisition": {
        "AcquisitionDate": "2023-03-09",
        "AcquisitionTime": "09:43:43",
        "FlightAltitude": "50 m above ground level",
        "FlightSpeed": "5 m/s"
    },

    "Platform": {
        "Platform": "DJI Matrice 300 RTK (M300) UAS",
        "SensorName": "DJI Zenmuse L1",
        "SensorType": "LiDAR"
    },

    "Processing": {
        "ProcessingLevel": "Level 1",
        "ProcessingDate": "22 Mar 2023",
        "SoftwareUsed": "DJI Terra software (version 3.5.0) and PDAL 2.8.4"
    },

    "Research": {
        "Investigator": "Laura Natali Sotomayor",
        "Contact": "laura.sotomayor@utas.edu.au",
        "Organisation": "University of Tasmania"
    },

    "Survey": {
        "FlightID": "TCATNM0006",
        "Operator": "University of Tasmania",
        "Description": (
            "LiDAR point cloud collected at TCATNM0006 (Beaufront) "
            "using the DJI Zenmuse L1 sensor with a Matrice 300 RTK (M300) UAS "
            "at a flying height of 50 m above ground level and a flying speed of 5 m/s."
        )
    },

    "Product": {
        "FileFormat": "COPC LAZ",
        "CRS": "EPSG:7855 + EPSG:5711",
        "Description": (
            "LiDAR point cloud converted to COPC with GDA2020 / MGA zone 55 "
            "+ AHD height."
        )
    }
}

In [12]:
meta = {
    "AcquisitionDate": "2023-03-09",
    "AcquisitionTime": "09:43:43",
    "DataProvider": "University of Tasmania",
    "Description": (
        "This is a LiDAR point cloud collected at TCATNM0006 (Beaufront) "
        "using the DJI Zenmuse L1 sensor with a Matrice 300 (M300) RTK UAS "
        "at a flying height of 50 m above ground level and a flying speed of 5 m/s."
    ),
    "FlightAltitude": "50 m above ground level",
    "FlightID": "TCATNM0006",
    "FlightSpeed": "5 m/s",
    "Investigator": "laura.sotomayor@utas.edu.au",
    "Organisation": "University of Tasmania",
    "Platform": "DJI Matrice 300 (M300) RTK UAS",
    "ProcessingDate": "22 Mar 2023",
    "ProcessingLevel": "Level 1",
    "ProjectName": "TerraLuma Research Lab",
    "SensorName": "DJI Zenmuse L1",
    "SensorType": "LiDAR",
    "SiteLocation": "TCATNM0006 (Beaufront)",
    "SoftwareUsed": "DJI Terra software (version 3.5.0) and PDAL 2.8.4",
    "CRS": "EPSG:7855 + EPSG:5711",
    "FileFormat": "COPC LAZ"
}

In [15]:
meta = {
    "AcquisitionDate": "2023-03-09",
    "AcquisitionTime": "09:43:43",
    "DataProvider": "University of Tasmania",
    "Description": (
        "This is a LiDAR point cloud collected at TCATNM0006 (Beaufront) "
        "using the DJI Zenmuse L1 sensor with a Matrice 300 (M300) RTK UAS "
        "at a flying height of 50 m above ground level and a flying speed of 5 m/s."
    ),
    "FlightAltitude": "50 m above ground level",
    "FlightID": "TCATNM0006",
    "FlightSpeed": "5 m/s",
    "Investigator": "Laura Natali Sotomayor",
    "Contact": "laura.sotomayor@utas.edu.au",
    "Organisation": "University of Tasmania",
    "Platform": "DJI Matrice 300 (M300) RTK UAS",
    "ProcessingDate": "22 Mar 2023",
    "ProcessingLevel": "Level 1",
    "ProjectName": "TerraLuma Research Lab",
    "SensorName": "DJI Zenmuse L1",
    "SensorType": "LiDAR",
    "SiteLocation": "TCATNM0006 (Beaufront)",
    "SoftwareUsed": "DJI Terra software (version 3.5.0) and PDAL 2.8.4",
    "CRS": "EPSG:7855 + EPSG:5711",
    "FileFormat": "COPC LAZ"
}

In [ ]:
# 1️⃣ Save your metadata JSON
import json

meta = {
    "AcquisitionDate": "2023-03-09",
    "AcquisitionTime": "09:43:43",
    "DataProvider": "University of Tasmania",
    "Description": (
        "This is a LiDAR point cloud collected at TCATNM0006 (Beaufront) "
        "using the DJI Zenmuse L1 sensor with a Matrice 300 (M300) RTK UAS "
        "at a flying height of 50 m above ground level and a flying speed of 5 m/s."
    ),
    "FlightAltitude": "50 m above ground level",
    "FlightID": "TCATNM0006",
    "FlightSpeed": "5 m/s",
    "Investigator": "Laura Natali Sotomayor",
    "Contact": "laura.sotomayor@utas.edu.au",
    "Organisation": "University of Tasmania",
    "Platform": "DJI Matrice 300 (M300) RTK UAS",
    "ProcessingDate": "22 Mar 2023",
    "ProcessingLevel": "Level 1",
    "ProjectName": "TerraLuma Research Lab",
    "SensorName": "DJI Zenmuse L1",
    "SensorType": "LiDAR",
    "SiteLocation": "TCATNM0006 (Beaufront)",
    "SoftwareUsed": "DJI Terra software (version 3.5.0) and PDAL 2.8.4",
    "CRS": "EPSG:7855 + EPSG:5711",
    "FileFormat": "COPC LAZ"
}
    "SensorType": "LiDAR",
    "SiteLocation": "TCATNM0006 (Beaufront)",
    "SoftwareUsed": "DJI Terra software (version 3.5.0) and PDAL 2.8.4",
    "CRS": "EPSG:7855 + EPSG:5711",
    "FileFormat": "COPC LAZ"
}

with open("beaufront_metadata.json", "w") as f:
    json.dump(meta, f, indent=4)

In [ ]:
# 2️⃣ PDAL pipeline to inject metadata
import pdal
import json

pipeline = {
    "pipeline": [
        "/data/beaufront/AHD/20230310_TCATNM0006.copc.laz",

        {
            "type": "writers.copc",
            "filename": "/data/beaufront/AHD/20230310_TCATNM0006_metadata.copc.laz",
            "a_srs": "EPSG:7855+5711",

            "system_id": "UTAS TerraLuma",
            "software_id": "PDAL 2.8.4 + FVC Structure Pipeline",

            "vlrs": [
                {
                    "user_id": "UTAS_TERRALUMA",
                    "record_id": 1,
                    "description": "TCATNM0006 UAS-LiDAR survey metadata (UTAS TerraLuma)",
                    "data": "beaufront_metadata.json"
                }
            ]
        }
    ]
}

p = pdal.Pipeline(json.dumps(pipeline))
p.execute()

78833222

In [ ]:
pdal pipeline add_metadata_pipeline.json